Create Schema/Database

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS my_coffee_shop
COMMENT 'Bright Coffee Shop analytics schema';


Database.schema.table

01.Create the dim_customers table.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS my_coffee_shop.dim_customers
(
 customer_id BIGINT PRIMARY KEY,
 first_name STRING,
 last_name STRING,
 email STRING,
 tier STRING,
 loyalty_points INT
)


02.Create the dim_products tables

In [0]:
%sql
CREATE TABLE IF NOT EXISTS my_coffee_shop.dim_products
(
    Product_id BIGINT PRIMARY KEY,
    product_code INT,
    product_name STRING,
    price DECIMAL(10,2),
    category STRING
)



03.Create the fact_orders table, partitioned by order_date.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS my_coffee_shop.fact_orders
(
    order_id BIGINT PRIMARY KEY,
    customer_id BIGINT,
    product_id BIGINT,
    store_id BIGINT,
    order_date DATE,
    quantity INT,
    unit_price DECIMAL(10,2),
    total_amount DECIMAL(10,2)
)
USING DELTA
PARTITIONED BY (order_date)


Create backup

In [0]:
%sql
Create table my_coffee_shop.fact_orders_backup
AS
SELECT * FROM my_coffee_shop.fact_orders;

num_affected_rows,num_inserted_rows


04.Enable column mapping on dim_customers, then rename loyalty_points to points.

In [0]:
%sql
ALTER TABLE my_coffee_shop.dim_customers SET TBLPROPERTIES('delta.columnMapping.mode' = 'name')

Rename column

In [0]:
%sql
ALTER TABLE my_coffee_shop.dim_customers RENAME COLUMN loyalty_points TO points

Describe table

In [0]:
%sql
DESCRIBE TABLE my_coffee_shop.dim_customers;

col_name,data_type,comment
customer_id,bigint,null
first_name,string,null
last_name,string,null
email,string,null
tier,string,null
points,int,null


** DATA MANIPULATION LANGUAGE**

05.Insert at least five customers into dim_customers.

In [0]:
%sql
insert into my_coffee_shop.dim_customers
(customer_id, first_name, last_name, email, tier, points)
values
(101, 'Lucy','Robyns','robynslucy@gmail.com','platinum', 700),
(102, 'Stan','Nze','stannze@gmail.com','gold', 500),
(103, 'Mel','Pow','powmel@gmail.com','silver', 650),
(104, 'Ted','Bundy','tedbundy@gmail.com','bronze', 800),
(105, 'Cathy','Jones','cjones@gmail.com','bronze', 750)


num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
SELECT * FROM my_coffee_shop.dim_customers;

customer_id,first_name,last_name,email,tier,points
101,Lucy,Robyns,robynslucy@gmail.com,platinum,700
102,Stan,Nze,stannze@gmail.com,gold,500
103,Mel,Pow,powmel@gmail.com,silver,650
104,Ted,Bundy,tedbundy@gmail.com,bronze,800
105,Cathy,Jones,cjones@gmail.com,bronze,750


06.Insert at least five products into dim_products

In [0]:
%sql
Insert into my_coffee_shop.dim_products
(product_id, product_code, product_name, category, price)
values
(1, '015', 'Coffee', 'Beverages', 5.99),
(2, '024', 'Tea', 'Beverages', 4.99),
(3, '033', 'Blueberry Muffin', 'Bakery', 2.99),
(4, '042', 'Banana Nut Muffin','Bakery', 1.99),
(5, '051', 'Chocolate Chip Muffin', 'Bakery', 3.99)


num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
SELECT *
FROM my_coffee_shop.dim_products;

Product_id,product_code,product_name,price,category
1,15,Coffee,5.99,Beverages
2,24,Tea,4.99,Beverages
3,33,Blueberry Muffin,2.99,Bakery
4,42,Banana Nut Muffin,1.99,Bakery
5,51,Chocolate Chip Muffin,3.99,Bakery


07.Update all Bronze-tier customers: add 50 to their points.

In [0]:
%sql
SELECT *
FROM my_coffee_shop.dim_customers
WHERE tier = 'bronze';

customer_id,first_name,last_name,email,tier,points
104,Ted,Bundy,tedbundy@gmail.com,bronze,800
105,Cathy,Jones,cjones@gmail.com,bronze,750


In [0]:
%sql
UPDATE  my_coffee_shop.dim_customers 
SET points = points + 50
WHERE tier = 'bronze';

num_affected_rows
2


In [0]:
%sql
SELECT *
FROM my_coffee_shop.dim_customers
WHERE tier = 'bronze';

customer_id,first_name,last_name,email,tier,points
104,Ted,Bundy,tedbundy@gmail.com,bronze,850
105,Cathy,Jones,cjones@gmail.com,bronze,800


08.back up dim_customers, then delete one customer by their customer_id.

In [0]:
%sql
CREATE TABLE dim_customers_backup
AS
SELECT *
FROM my_coffee_shop.dim_customers;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM my_coffee_shop.dim_customers
WHERE customer_id = 102

customer_id,first_name,last_name,email,tier,points
102,Stan,Nze,stannze@gmail.com,gold,500


In [0]:
%sql
DELETE FROM my_coffee_shop.dim_customers
WHERE customer_id = 102;

num_affected_rows
1


In [0]:
%sql
SELECT *
FROM my_coffee_shop.dim_customers;


customer_id,first_name,last_name,email,tier,points
101,Lucy,Robyns,robynslucy@gmail.com,platinum,700
103,Mel,Pow,powmel@gmail.com,silver,650
104,Ted,Bundy,tedbundy@gmail.com,bronze,850
105,Cathy,Jones,cjones@gmail.com,bronze,800


In [0]:
%sql
DESCRIBE HISTORY my_coffee_shop.dim_customers;


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
7,2026-06-01T16:34:36.000Z,76313986132621,primmkwesha@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(861371086995787),3a75e4e7-4c0f-4431-a670-fca4943110be,0601-151742-a6240bpt-v2n,6,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 3411, p25FileSize -> 3380, numDeletionVectorsRemoved -> 1, minFileSize -> 3380, numAddedFiles -> 1, maxFileSize -> 3380, p75FileSize -> 3380, p50FileSize -> 3380, numAddedBytes -> 3380)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
6,2026-06-01T16:34:34.000Z,76313986132621,primmkwesha@gmail.com,DELETE,"Map(predicate -> [""(customer_id#13763L = 102)""])",null,List(861371086995787),3a75e4e7-4c0f-4431-a670-fca4943110be,0601-151742-a6240bpt-v2n,5,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1378, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 930, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 447)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
5,2026-06-01T16:23:31.000Z,76313986132621,primmkwesha@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(861371086995787),7e34b8c3-461f-4801-84cf-f56841e4d4ea,0601-151742-a6240bpt-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 5765, p25FileSize -> 3411, numDeletionVectorsRemoved -> 1, minFileSize -> 3411, numAddedFiles -> 1, maxFileSize -> 3411, p75FileSize -> 3411, p50FileSize -> 3411, numAddedBytes -> 3411)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
4,2026-06-01T16:23:28.000Z,76313986132621,primmkwesha@gmail.com,UPDATE,"Map(predicate -> [""(tier#12697 = bronze)""])",null,List(861371086995787),7e34b8c3-461f-4801-84cf-f56841e4d4ea,0601-151742-a6240bpt-v2n,3,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 4672, numDeletionVectorsUpdated -> 0, scanTimeMs -> 2552, numAddedFiles -> 1, numUpdatedRows -> 2, numAddedBytes -> 3189, rewriteTimeMs -> 2087)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
3,2026-06-01T15:52:15.000Z,76313986132621,primmkwesha@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(861371086995787),52faa59e-a4fb-4313-ae05-524f55c2de01,0601-151742-a6240bpt-v2n,2,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 5, numOutputBytes -> 2576)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
2,2026-06-01T15:18:08.000Z,76313986132621,primmkwesha@gmail.com,RENAME COLUMN,"Map(oldColumnPath -> loyalty_points, newColumnPath -> points)",null,List(861371086995787),4c3bab9a-ba42-4969-aed0-eb3046d4169a,0601-151742-a6240bpt-v2n,1,WriteSerializable,true,Map(),null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
1,2026-06-01T14:00:01.000Z,76313986132621,primmkwesha@gmail.com,SET TBLPROPERTIES,"Map(properties -> {""delta.columnMapping.mode"":""name""})",null,List(861371086995787),183e808f-2fbf-47c0-ab33-4032aa39484d,0601-135839-d1g0yudr-v2n,0,WriteSerializable,true,Map(),null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
0,2026-06-01T12:10:52.000Z,76313986132621,primmkwesha@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-fe04e4d9-e454-4c3d-97ec-05056a5f8bb7"",""delta.rowTracking.materializedRow